In [1]:
import sys; sys.path.append("..")

# 02c – So sánh PCA và FAMD, thêm t-SNE (#12)

Dùng lại luồng của `02_reduce`: `clean` → `add_derived_features` → chia 7:3 có phân tầng, seed 0. PCA và FAMD **chỉ fit trên train**, validation chỉ transform. Bảng so sánh tại 11 thành phần (1/3 của 33 chiều); t-SNE 2 chiều **chỉ để nhìn**, không dùng làm đầu vào mô hình. Hình PCA/FAMD riêng lẻ đã có ở `02_reduce`, không vẽ lại ở đây.

In [2]:
import numpy as np
import pandas as pd
from src.data import load_raw, TARGET
from src.preprocess import clean, add_derived_features, make_splits
from src import reduce as rd

pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 30)
K = 11
FIG = rd.FIG_DIR

## 0. Dữ liệu và chia train / validation

In [3]:
df = add_derived_features(clean(load_raw()))
y = df[TARGET]
tr_idx, va_idx = make_splits(y, [0], {'7:3': 0.3})[('7:3', 0)]
df_tr, df_va = df.loc[tr_idx], df.loc[va_idx]
y_tr, y_va = y.loc[tr_idx], y.loc[va_idx]
pd.DataFrame({'số mẫu': [len(df_tr), len(df_va)], 'tỉ lệ vỡ nợ': [y_tr.mean(), y_va.mean()]},
             index=['train', 'validation']).round(4)

EDUCATION = 0 -> 4: 14 mẫu
EDUCATION = 5 -> 4: 280 mẫu
EDUCATION = 6 -> 4: 51 mẫu
MARRIAGE = 0 -> 3: 54 mẫu
Loại bản ghi trùng hoàn toàn: 35 dòng (còn 29965)


,số mẫu,tỉ lệ vỡ nợ
train,20975,0.2213
validation,8990,0.2212


## 1. Bảng so sánh tại 11 thành phần

`compare_reductions` fit PCA (33 chiều sau one-hot) và FAMD (30 chiều hỗn hợp) trên train, rồi tính:
- **Explained variance**: tích lũy tại 11 thành phần, số thành phần để đạt 80% / 90%.
- **Khả năng tách lớp**: ROC-AUC đơn biến lớn nhất / trung bình của 11 thành phần (trên validation, bỏ qua dấu), và ROC-AUC / PR-AUC của LogisticRegression fit trên 11 thành phần của train, chấm trên validation.
- **Khả năng giải thích**: khối đặc trưng chi phối từng thành phần trong 3 thành phần đầu (tỉ trọng loading²), kèm khối đứng thứ hai để thấy khi nào hai khối gần ngang nhau.

In [4]:
tab, coords = rd.compare_reductions(df_tr, y_tr, df_va, y_va, k=K, return_coords=True)
from src.config import RESULTS_DIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
tab.to_csv(RESULTS_DIR / 'reduce_compare.csv', encoding='utf-8')
tab.T

method,PCA,FAMD
k,11,11
n_for_80%,11,11
n_for_90%,15,16
cum_var_k,0.817991,0.805561
auc_abs_max,0.736624,0.737329
best_comp,2,2
auc_abs_mean,0.560922,0.54509
lr_roc_auc,0.742827,0.743131
lr_pr_auc,0.484187,0.484722
top_block_1,BILL_AMT,BILL_AMT


Để so với mốc: LogisticRegression trên cả 33 chiều đã chuẩn hóa (không giảm chiều) chấm trên cùng validation.

In [5]:
from sklearn.linear_model import LogisticRegression
from src.metrics import classification_metrics
_, X_tr, X_va = rd.pca_input(df_tr, df_va)
base = classification_metrics(y_va, LogisticRegression(max_iter=1000).fit(X_tr, y_tr).predict_proba(X_va)[:, 1])
pd.DataFrame({'33 chiều gốc': base}).loc[['roc_auc', 'pr_auc']].round(4).T

,roc_auc,pr_auc
33 chiều gốc,0.7498,0.4997


## 2. t-SNE 2 chiều (chỉ để trực quan hóa)

t-SNE không có phép transform cho điểm mới, và khoảng cách giữa các cụm trên hình không có nghĩa định lượng, nên **không** dùng làm đầu vào mô hình; chỉ để nhìn xem nhãn có tách theo cấu trúc cục bộ hay không. Lấy ngẫu nhiên 5000 điểm train (seed 0); dữ liệu rộng hơn 30 chiều được PCA xuống 30 chiều trước. Ba đầu vào: 33 chiều chuẩn hóa (raw-scaled), 11 thành phần PCA, 11 thành phần FAMD.

In [6]:
Z_pca, Z_famd = coords['PCA'][0][:, :K], coords['FAMD'][0][:, :K]
paths = [
    rd.plot_tsne(X_tr.to_numpy(), y_tr, FIG / 'tsne_raw.png', title='t-SNE – 33 chiều chuẩn hóa'),
    rd.plot_tsne(Z_pca, y_tr, FIG / 'tsne_pca.png', title=f't-SNE – {K} thành phần PCA'),
    rd.plot_tsne(Z_famd, y_tr, FIG / 'tsne_famd.png', title=f't-SNE – {K} thành phần FAMD'),
]
paths

[WindowsPath('C:/Users/chous/Documents/coding every thing (from OneDrive - komodore)/hus/ml/ml-credit-default/results/figures/tsne_raw.png'),
 WindowsPath('C:/Users/chous/Documents/coding every thing (from OneDrive - komodore)/hus/ml/ml-credit-default/results/figures/tsne_pca.png'),
 WindowsPath('C:/Users/chous/Documents/coding every thing (from OneDrive - komodore)/hus/ml/ml-credit-default/results/figures/tsne_famd.png')]

![](../results/figures/tsne_raw.png) ![](../results/figures/tsne_pca.png) ![](../results/figures/tsne_famd.png)

## 3. Nhận xét

**Explained variance.** Hai phương pháp gần như trùng nhau: 11 thành phần giữ 81,8% (PCA) và 80,6% (FAMD), cùng cần 11 thành phần để đạt 80%, và 15 so với 16 để đạt 90%. Em chưa so được "hơn kém" ở đây vì thang đo khác loại: PCA là phương sai của 33 chiều đã one-hot, FAMD là quán tính của 30 chiều hỗn hợp, nên chênh 1,2 điểm phần trăm này không nói lên PCA tốt hơn.

**Khả năng tách lớp.** Thành phần tách tốt nhất ở cả hai là thành phần 2 với AUC đơn biến khoảng 0,737, còn trung bình 11 thành phần chỉ 0,55–0,56, tức hầu hết thành phần gần như không liên quan đến nhãn. LogisticRegression trên 11 thành phần cho ROC-AUC 0,743 (PCA) và 0,743 (FAMD), PR-AUC 0,484 và 0,485; chênh nhau dưới 0,001 nên coi như bằng nhau (chỉ một lần chia, seed 0, chưa có khoảng tin cậy). Mốc 33 chiều gốc đạt 0,750 / 0,500, cao hơn một chút, nghĩa là bỏ 22 chiều làm mất một ít thông tin dự đoán dù giữ hơn 80% phương sai. Phương sai lớn không đồng nghĩa với liên quan đến nhãn.

**Khả năng giải thích.** Hai thành phần đầu có cùng ý nghĩa ở cả hai phương pháp: thành phần 1 do khối BILL_AMT chi phối (~42%, khối thứ hai là PAY ~29%). Thành phần 2 thì PAY_AMT (~35%) và PAY gần ngang nhau (33% ở PCA, 34% ở FAMD), nên em không dám nói thành phần 2 "là PAY_AMT" một cách chắc chắn; vì thành phần 2 cũng là thành phần tách lớp tốt nhất (AUC 0,737), em đoán nhờ phần PAY (trạng thái trả nợ) hơn là PAY_AMT, nhưng chỉ là suy đoán từ tỉ trọng loading, chưa kiểm tra riêng. Khác biệt xuất hiện ở thành phần 3: PCA gần như chỉ là khối "Hồ sơ" (99%, tức các cột one-hot SEX/EDUCATION/MARRIAGE đứng riêng một trục), còn FAMD là khối "Dẫn xuất" (52%, khối thứ hai là PAY ~29%) vì FAMD gộp category vào một cách mã hóa kiểu MCA thay vì để mỗi mức one-hot kéo thành một trục. Ở điểm này FAMD dễ đọc hơn một chút, nhưng loading của category trong FAMD là tọa độ mức chứ không phải tương quan, nên chỉ so sánh tương đối.

**t-SNE (chỉ để nhìn).** Cả ba hình đều không tách hai lớp: điểm vỡ nợ (đỏ) lẫn trong điểm không vỡ nợ, chỉ có vài vùng nghiêng về đỏ (cụm bên phải của hình raw-scaled, vùng phía trên bên trái của hình FAMD). Hình PCA và FAMD trông giống nhau về mức độ lẫn lớp, khớp với bảng số ở trên. Em không đọc gì từ khoảng cách hay kích thước cụm vì t-SNE không bảo toàn chúng, và mỗi hình chỉ là 5000 điểm với một seed.

**Kết luận tạm.** Với bộ dữ liệu này PCA và FAMD cho kết quả gần như tương đương; FAMD có lợi thế nhỏ ở khả năng giải thích do xử lý category trực tiếp, còn PCA đơn giản hơn. Việc chọn cái nào để đưa vào mô hình nên dựa vào kết quả K-scenario thật, không dựa riêng vào bảng này.